# 🎙️ ZeroTTS Studio (TSS) - Google Colab Free Tier
**Mô hình Text-to-Speech (TTS) Tiếng Việt với toàn bộ tính năng tùy chỉnh: Phân đoạn tag `[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`, gộp `_FULL_MERGED.mp3`, Streaming Audio.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### ⚡ Hướng dẫn cài đặt nhanh (1-Click Run):
1. **Bật GPU T4 (Khuyên dùng)**: Vào menu **Runtime (Thời gian chạy)** -> **Change runtime type (Thay đổi loại thời gian chạy)** -> Chọn **T4 GPU** -> Nhấn **Save (Lưu)**.
2. **Chạy tuần tự các bước**: 
   - **Bước 1**: Kết nối Google Drive (để lưu file audio vĩnh viễn và nạp mã nguồn).
   - **Bước 2**: Nạp mã nguồn tùy chỉnh của bạn (`TSS_Code.zip`, Google Drive, hoặc GitHub cá nhân).
   - **Bước 3**: Tải Model Weights từ Hugging Face.
   - **Bước 4**: Bấm vào link **Public URL** (`https://xxxx.trycloudflare.com`) để mở WebUI Studio!

## ⚙️ Bước 1: Kiểm tra GPU & Kết nối Google Drive
> *Google Drive giúp tự động lưu toàn bộ file âm thanh đã xuất (`outputs/`) và cho phép nạp mã nguồn `TSS_Code.zip` một cách thuận tiện.*

In [ ]:
#@title Cấu hình Lưu Trữ & Kiểm tra Phần Cứng { run: "auto", display-mode: "form" }
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ZeroTTS_Outputs"

import os
import subprocess

# 1. Kiem tra GPU
print("🔍 Đang kiểm tra phần cứng...")
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8").strip()
    print(f"✅ Phát hiện GPU: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("ℹ️ Đang chạy trên CPU. Bạn có thể bật GPU tại: Runtime -> Change runtime type -> T4 GPU.")
    HAS_GPU = False

# 2. Mount Google Drive nếu được bật
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        os.environ["ZEROTTS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
        print(f"📁 File âm thanh sẽ được lưu vĩnh viễn tại Google Drive:")
        print(f"   -> {DRIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể kết nối Google Drive ({e}). Sử dụng thư mục tạm /content/outputs.")
        os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
else:
    os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
    print("📁 Dữ liệu sẽ lưu tạm tại /content/outputs (sẽ bị xoá khi tắt Colab).")

## 📦 Bước 2: Nạp Mã Nguồn Tùy Chỉnh Của Bạn & Cài Đặt Thư Viện
Chọn một trong các nguồn mã nguồn bên dưới:
- **`TSS_Code.zip từ Google Drive`**: Đặt file `TSS_Code.zip` trong Google Drive (`MyDrive/TSS_Code.zip`).
- **`Tải lên trực tiếp TSS_Code.zip`**: Bấm upload file zip 7.5MB trực tiếp từ máy tính của bạn.
- **`Thư mục có sẵn trên Drive`**: Nếu bạn đã sao chép nguyên thư mục `TSS-main` vào Drive (`MyDrive/TSS-main`).
- **`Clone từ GitHub`**: Nếu bạn đã đưa code lên repository GitHub cá nhân.

In [ ]:
#@title Chọn Phương Thức Nạp Mã Nguồn { run: "auto", display-mode: "form" }
SOURCE_MODE = "Tu dong tim TSS_Code.zip (Drive hoac Upload)" #@param ["Tu dong tim TSS_Code.zip (Drive hoac Upload)", "Thu muc co san tren Google Drive (/content/drive/MyDrive/TSS-main)", "Git Clone tu GitHub Repo ca nhan"]
GITHUB_REPO_URL = "" #@param {type:"string"}
DRIVE_SOURCE_PATH = "/content/drive/MyDrive/TSS-main" #@param {type:"string"}

import os
import shutil
import zipfile

APP_DIR = "/content/TSS"
os.makedirs("/content", exist_ok=True)

if SOURCE_MODE == "Tu dong tim TSS_Code.zip (Drive hoac Upload)":
    # 1. Tim trong Drive
    drive_zip = "/content/drive/MyDrive/TSS_Code.zip"
    uploaded_zip = "/content/TSS_Code.zip"
    target_zip = None
    
    if os.path.isfile(drive_zip):
        print(f"✅ Đã tìm thấy file mã nguồn tùy chỉnh trong Google Drive: {drive_zip}")
        target_zip = drive_zip
    elif os.path.isfile(uploaded_zip):
        print(f"✅ Đã tìm thấy file mã nguồn tải lên: {uploaded_zip}")
        target_zip = uploaded_zip
    else:
        print("📤 Chưa tìm thấy file zip. Vui lòng bấm 'Choose Files' bên dưới để tải file TSS_Code.zip từ máy của bạn lên:")
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                target_zip = os.path.join("/content", fname)
                break
    
    if target_zip and os.path.isfile(target_zip):
        print(f"📦 Đang giải nén mã nguồn {target_zip} vào {APP_DIR}...")
        shutil.rmtree(APP_DIR, ignore_errors=True)
        with zipfile.ZipFile(target_zip, 'r') as zf:
            zf.extractall(APP_DIR)
        print("✅ Đã nạp thành công toàn bộ mã nguồn tùy chỉnh của bạn!")
    else:
        raise FileNotFoundError("❌ Không tìm thấy file zip mã nguồn. Vui lòng kiểm tra lại!")

elif SOURCE_MODE == "Thu muc co san tren Google Drive (/content/drive/MyDrive/TSS-main)":
    if os.path.isdir(DRIVE_SOURCE_PATH):
        print(f"✅ Sử dụng mã nguồn trực tiếp từ Google Drive: {DRIVE_SOURCE_PATH}")
        shutil.rmtree(APP_DIR, ignore_errors=True)
        shutil.copytree(DRIVE_SOURCE_PATH, APP_DIR, ignore=shutil.ignore_patterns('.venv', 'python-3.11*', 'ZeroTTS_model', 'outputs'))
    else:
        raise FileNotFoundError(f"❌ Không tìm thấy thư mục {DRIVE_SOURCE_PATH} trên Google Drive!")

elif SOURCE_MODE == "Git Clone tu GitHub Repo ca nhan":
    if GITHUB_REPO_URL:
        shutil.rmtree(APP_DIR, ignore_errors=True)
        !git clone $GITHUB_REPO_URL $APP_DIR
    else:
        raise ValueError("❌ Vui lòng nhập URL GitHub Repo cá nhân của bạn!")

%cd $APP_DIR

print("\n🔧 Đang cài đặt thư viện hệ thống và Python (chỉ mất ~1 phút)...")
!apt-get update -qq && apt-get install -y -qq ffmpeg libportaudio2

if HAS_GPU:
    print("⚡ Đang cài đặt ONNX Runtime GPU (Tăng tốc CUDA T4)...")
    !pip install -q onnxruntime-gpu>=1.17.0
else:
    print("⚙️ Đang cài đặt ONNX Runtime CPU...")
    !pip install -q onnxruntime>=1.17.0

!pip install -q soundfile sounddevice fastapi uvicorn tokenizers huggingface_hub scipy requests pydantic
!pip install -q --no-deps -e .

print("\n✅ Cài đặt hoàn tất! Toàn bộ tính năng (Tag timeline, gộp MP3, WebUI) đã sẵn sàng.")

## 🧠 Bước 3: Tải Model Weights (Mô Hình Pre-trained)

In [ ]:
#@title Tải Model zeroweight-ai/ZeroTTS từ Hugging Face
from huggingface_hub import snapshot_download
import shutil

MODEL_DIR = os.path.join(APP_DIR, "ZeroTTS_model")
local_voices_dir = os.path.join(MODEL_DIR, "voices")
custom_voices_backup = "/content/custom_voices_backup"

# 1. Bảo lưu toàn bộ các voice tùy chỉnh từ mã nguồn của bạn
has_custom_voices = os.path.exists(local_voices_dir) and len(os.listdir(local_voices_dir)) > 0
if has_custom_voices:
    shutil.rmtree(custom_voices_backup, ignore_errors=True)
    shutil.copytree(local_voices_dir, custom_voices_backup)
    print(f"📦 Đã tìm thấy {len(os.listdir(local_voices_dir))} gói giọng tùy chỉnh từ máy của bạn (sẽ bảo lưu 100%).")

# 2. Tải ONNX weights nếu chưa có
onnx_check = os.path.join(MODEL_DIR, "onnx", "prefix_step.onnx")
if not os.path.exists(onnx_check):
    print("⏳ Đang tải mô hình ZeroTTS từ Hugging Face (~500MB)... Vui lòng chờ 10-30 giây.")
    snapshot_download(repo_id="zeroweight-ai/ZeroTTS", local_dir=MODEL_DIR)
    print("✅ Đã tải xong Model Weights!")
else:
    print("✅ Mô hình đã có sẵn trong ZeroTTS_model!")

# 3. Khôi phục các voice tùy chỉnh từ máy của bạn (đảm bảo đồng nhất 100% với Local)
if has_custom_voices and os.path.exists(custom_voices_backup):
    for item in os.listdir(custom_voices_backup):
        src_path = os.path.join(custom_voices_backup, item)
        dst_path = os.path.join(local_voices_dir, item)
        if os.path.isdir(src_path):
            shutil.rmtree(dst_path, ignore_errors=True)
            shutil.copytree(src_path, dst_path)
        else:
            shutil.copy2(src_path, dst_path)
    print("✅ Đã đồng bộ chính xác toàn bộ gói giọng từ máy local của bạn!")


## 🚀 Bước 4: Khởi Chạy WebUI với Cloudflare Tunnel (Miễn phí 100%)
> *Cloudflare Tunnel sẽ tạo một đường link Public HTTPS an toàn (`https://xxxx.trycloudflare.com`) để bạn truy cập WebUI từ bất kỳ thiết bị nào.*

In [ ]:
#@title Khởi chạy WebUI Studio & Mở Public URL { display-mode: "form" }
import subprocess
import time
import re
import urllib.request
import os

# 1. Download cloudflared binary nếu chưa có
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📥 Đang tải Cloudflare Tunnel (cloudflared)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 2. Chạy FastAPI Server ở chế độ nền
print("🚀 Đang khởi động ZeroTTS WebUI Server...")
server_cmd = ["python", "webui/server.py", "--model", MODEL_DIR, "--host", "0.0.0.0", "--port", "7860"]
server_proc = subprocess.Popen(server_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(3)

# 3. Khởi động Cloudflare Tunnel
print("🌐 Đang mở đường hầm Cloudflare Tunnel...")
tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 65)
if public_url:
    print(f"🎉 WEBUI ĐÃ SẴN SÀNG! HÃY BẤM VÀO ĐƯỜNG LINK DƯỚI ĐÂY:")
    print(f"👉 Public URL: {public_url}")
else:
    print("⚠️ Đang khởi động tunnel, vui lòng kiểm tra lại log bên dưới.")
print("=" * 65)

# Giữ tiến trình hoạt động và hiển thị log
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng WebUI Server.")
    server_proc.terminate()
    tunnel_proc.terminate()

## ⚡ Bước 5: Chế Độ Dòng Lệnh (CLI Batch Render)
> *Sinh âm thanh hàng loạt trực tiếp trong Colab từ văn bản có gắn thẻ (`[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`, `[Kết Thúc]`) mà không cần mở WebUI.*

In [ ]:
#@title Chạy Render Hàng Loạt Bằng Script { display-mode: "form" }
sample_input_text = """[Câu 1]
Xin chào các bạn. [pause: 1.5s] Đây là câu hỏi đầu tiên.

[Câu 2]
Hãy chọn đáp án đúng nhất. Thời gian suy nghĩ bắt đầu!

#[Bỏ qua]
Đoạn này nháp, hệ thống tự động bỏ qua không đọc.

[Kết Thúc]
Chúc các bạn làm bài tốt!
"""

VOICE_NAME = "nam-mien-bac" #@param ["nam-mien-bac", "nu-mien-bac", "nam-mien-nam", "nu-mien-nam", "unconditional"]
OUTPUT_PROJECT_NAME = "du_an_colab_01" #@param {type:"string"}

import webui.engine as engine
import os

engine.set_model(MODEL_DIR)

print(f"🎯 Đang xử lý kịch bản cho dự án: {OUTPUT_PROJECT_NAME}...")
blocks = engine.parse_input_tags(sample_input_text)

print(f"📋 Tổng số phân đoạn nhận diện: {len(blocks)}")
for b in blocks:
    status = "⏭️ BỎ QUA" if b["is_skipped"] else "✅ RENDER"
    print(f" - [{b['tag']}]: {status} | Nội dung: {b['text'][:40]}...")

# Chạy tổng hợp
voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME
mode_param = "uncond" if VOICE_NAME == "unconditional" else "voice"

print("\n🔊 Đang tiến hành sinh âm thanh và gộp file...")
out_folder = None
for event_data in engine.generate_from_parsed_blocks(
    blocks=blocks,
    voice_name=voice_param,
    mode=mode_param,
    custom_name=OUTPUT_PROJECT_NAME,
    auto_concat=True,
    merged_format="MP3"
):
    if "status" in event_data:
        print(f"   {event_data['status']}")
    if "out_folder" in event_data:
        out_folder = event_data["out_folder"]

print(f"\n🎉 Hoàn thành! Thư mục kết quả: {out_folder}")
if out_folder and os.path.exists(out_folder):
    print("📂 Danh sách file đã tạo:")
    for f in sorted(os.listdir(out_folder)):
        fpath = os.path.join(out_folder, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  ├── {f} ({size_kb:.1f} KB)")